In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F # for jitter we need this
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import itertools
import csv
import os
# Device configuration - Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
model = models.googlenet(pretrained=True).to(device)
# Print model architecture for debugging and layer inspection
# print("Model architecture:")
# print(model)
# Freeze all model parameters since we'll only be modifying the input image
for param in model.parameters():
    param.requires_grad = False

c:\Users\Anurath\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Anurath\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
# utilities from the notebook - no changes here

##### var loss function
def total_variation_loss(img):
    # Calculate differences between adjacent pixels horizontally and vertically
    tv_loss = torch.sum(torch.abs(img[:, :, :-1] - img[:, :, 1:])) + \
              torch.sum(torch.abs(img[:, :-1, :] - img[:, 1:, :]))
    return tv_loss
####### Image prociessing and deprocessing
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])
def deprocess(image_tensor):
    image = image_tensor.clone().detach().cpu()
    # Reverse the normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    image = image * std + mean
    # Ensure pixel values are in valid range [0,1]
    image = image.clamp(0,1)
    image = image.squeeze(0)  # Remove batch dimension
    image = image.permute(1,2,0)  # C x H x W => H x W x C
    return image.numpy()

###### Jitter add and remove utils
def add_random_jitter(image, jitter=32):
    b, c, h, w = image.shape
    
    # Pad the image with reflect padding
    padded = F.pad(image, (jitter, jitter, jitter, jitter), mode='reflect')
    
    # Generate random crop coordinates
    dx = torch.randint(0, jitter * 2, (1,))
    dy = torch.randint(0, jitter * 2, (1,))
    # Crop the image at the random offset
    jittered = padded[:, :, dy:dy + h, dx:dx + w]
    
    return jittered, (dx, dy)
def remove_jitter(image, jitter_amounts, orig_shape):
    dx, dy = jitter_amounts
    h, w = orig_shape[-2:]
    return image[:, :, :h, :w]

In [4]:
# Modified deep_dream function with jitter - no changes
def deep_dream_with_jitter(model, image, iterations, lr, layer_names_weights, optimizer_name='Adam', tv_weight=0.0000001, jitter=32):
    orig_shape = image.shape
    image = image.clone().requires_grad_(True).to(device)

    activations = {}
    hooks = []

    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output
        return hook

    for name, module in model.named_modules():
        if name in layer_names_weights:
            hooks.append(module.register_forward_hook(get_activation(name)))

    if optimizer_name == 'Adam':
        optimizer = optim.Adam([image], lr=lr)
        use_closure = False
    elif optimizer_name == 'LBFGS':
        optimizer = optim.LBFGS([image], lr=lr, max_iter=iterations)
        use_closure = True
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD([image], lr=lr, momentum=0.9)
        use_closure = False

    if use_closure:
        iteration = [0]

        def closure():
            optimizer.zero_grad()
            activations.clear()
            
            # Add jitter
            jittered_image, jitter_amounts = add_random_jitter(image, jitter)
            
            out = model(jittered_image)
            losses = []
            for name in layer_names_weights:
                if name in activations:
                    weight = layer_names_weights[name]
                    # activation = activations[name]
                    activation = activations[name] ** 2
                    # losses.append(-weight * activation.norm())
                    losses.append(-weight * activation.mean())
            
            dream_loss = sum(losses)
            tv_loss = total_variation_loss(jittered_image[0])
            loss = dream_loss + tv_weight * tv_loss
            
            loss.backward()
            
            # Remove jitter from gradients
            if image.grad is not None:
                image.grad.data = remove_jitter(image.grad.data, jitter_amounts, orig_shape)

            # if iteration[0] % 25 == 0:
            #     print(f"Iteration {iteration[0]}, Loss: {loss.item()}")

            iteration[0] += 1
            return loss

        optimizer.step(closure)
        image.data.clamp_(-1.5, 1.5)

    else:
        for i in range(iterations):
            optimizer.zero_grad()
            activations.clear()
            
            # Add jitter
            jittered_image, jitter_amounts = add_random_jitter(image, jitter)
            
            out = model(jittered_image)
            
            losses = []
            for name in layer_names_weights:
                if name in activations:
                    weight = layer_names_weights[name]
                    # activation = activations[name]
                    activation = activations[name] ** 2
                    # losses.append(-weight * activation.norm())
                    losses.append(-weight * activation.mean())
            
            dream_loss = sum(losses)
            tv_loss = total_variation_loss(jittered_image[0])
            loss = dream_loss + tv_weight * tv_loss
            
            loss.backward()
            
            # Remove jitter from gradients
            if image.grad is not None:
                image.grad.data = remove_jitter(image.grad.data, jitter_amounts, orig_shape)
            
            optimizer.step()
            image.data.clamp_(-1.5, 1.5)

            # if i % 25 == 0:
            #     print(f"Iteration {i}, Loss: {loss.item()}")

    for hook in hooks:
        hook.remove()

    return image.detach()

In [5]:
# multiscale deep dream - no changes
def deep_dream_multiscale_with_jitter(model, base_image, iterations, lr, layer_names_weights, 
                                    num_octaves, octave_scale, optimizer_name='Adam', jitter=32):
    image = base_image.clone()
    octaves = []

    for i in range(num_octaves):
        scale_factor = octave_scale ** (-i)
        size = [int(dim * scale_factor) for dim in image.shape[-2:]]
        octave_image = nn.functional.interpolate(image, size=size, mode='bilinear', align_corners=False)
        octaves.append(octave_image)
    
    detail = torch.zeros_like(octaves[-1], device=device)

    for octave, octave_image in enumerate(reversed(octaves)):
        # print(f"Processing octave {num_octaves - octave}")
        if detail.shape != octave_image.shape:
            detail = nn.functional.interpolate(detail, size=octave_image.shape[-2:], 
                                            mode='bilinear', align_corners=False)
        input_image = octave_image + detail
        
        # Use the jittered version of deep dream
        dreamed_image = deep_dream_with_jitter(model, input_image, iterations, lr, 
                                             layer_names_weights, optimizer_name, 
                                             jitter=max(int(jitter * octave_scale**(-octave)), 8))
        detail = dreamed_image - octave_image

    return dreamed_image

In [6]:
# results folder and csv
image = Image.open("images/konatsu_kato.jpg")
input_image = preprocess(image).unsqueeze(0).to(device)

os.makedirs("results", exist_ok=True)
csv_path = "results/experiments.csv"

with open(csv_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "experiment_id",
        "layers",
        "num_octaves",
        "octave_scale",
        "lr",
        "jitter",
        "optimizer",
        "image_filename"
    ])

In [7]:
# hyperparameters for experiment
layer_options = [
    {'inception4a': 1.0},
    {'inception4a': 1.0, 'inception4b': 1.0},
    {'inception4b': 1.0, 'inception4c': 1.0},
    {'inception4c': 1.0, 'inception4d': 1.0},
    {'inception4d': 1.0, 'inception4e': 2.0},
]

octave_options = [3, 5]
octave_scale_options = [1.2, 1.4]
learning_rates = [0.01, 0.05, 0.1]
jitters = [16, 32]
optimizers = ['Adam']
iterations = 50
experiment_id = 0

In [ ]:
# exhaustive loop
for layers, num_octaves, octave_scale, lr, jitter, opt in itertools.product(
    layer_options,
    octave_options,
    octave_scale_options,
    learning_rates,
    jitters,
    optimizers
):

    experiment_id += 1
    print("\n=============================================")
    print(f"Experiment {experiment_id}")
    print(f"Layers: {layers} Num Octaves: {num_octaves}, Octave Scale: {octave_scale}, LR: {lr}, Jitter: {jitter}, Optimizer: {opt}" )
    print("=============================================\n")

    dreamed_image = deep_dream_multiscale_with_jitter(model, input_image, iterations, lr, layers, num_octaves, octave_scale, optimizer_name=opt, jitter=jitter)
    result = deprocess(dreamed_image)
    
    filename = f"dream_{experiment_id}.png"
    save_path = os.path.join("results", filename)
    plt.imsave(save_path, result)
    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            experiment_id,
            str(layers),
            num_octaves,
            octave_scale,
            lr,
            jitter,
            opt,
            filename
        ])
        
    plt.figure(figsize=(5, 5))
    plt.imshow(result)
    plt.title(f"Experiment {experiment_id}")
    plt.axis("off")
    plt.show()

In [ ]:
# best result combinations
# 5 = {'inception4a': 1.0}, octaves = 3,scale = 1.2,lr=0.1,jit = 16,= opt = "Adam",
# 22 = {'inception4a': 1.0}, octaves = 5,scale = 1.4,lr=0.05,jit = 32,= opt = "Adam",
# 29 = "{'inception4a': 1.0, 'inception4b': 1.0}", octaves = 3,scale = 1.2,lr=0.1,jit = 16,= opt = "Adam",
# 59 = "{'inception4b': 1.0, 'inception4c': 1.0}", octaves = 3,scale = 1.4,lr=0.1,jit = 16,= opt = "Adam",
# 66 = "{'inception4b': 1.0, 'inception4c': 1.0}", octaves = 5,scale = 1.2,lr=0.1,jit = 32,= opt = "Adam",
# 47 = "{'inception4a': 1.0, 'inception4b': 1.0}", octaves = 5,scale = 1.4,lr=0.1,jit = 16,= opt = "Adam",
# 102 = "{'inception4d': 1.0, 'inception4e': 2.0}", octaves = 3,scale = 1.2,lr=0.1,jit = 32,= opt = "Adam",
# 107 = "{'inception4d': 1.0, 'inception4e': 2.0}", octaves = 3,scale = 1.4,lr=0.1,jit = 16,= opt = "Adam",
# 118 = "{'inception4d': 1.0, 'inception4e': 2.0}", octaves = 5,scale = 1.4,lr=0.05,jit = 32,= opt = "Adam",
# 120 = "{'inception4d': 1.0, 'inception4e': 2.0}", octaves = 5,scale = 1.4,lr=0.1,jit = 32,= opt = "Adam",

In [8]:
# New folder
image = Image.open("images/konatsu_kato.jpg")
input_image = preprocess(image).unsqueeze(0).to(device)

os.makedirs("new_results", exist_ok=True)
csv_path = "new_results/experiments.csv"

with open(csv_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "experiment_id",
        "layers",
        "num_octaves",
        "octave_scale",
        "lr",
        "jitter",
        "optimizer",
        "image_filename"
    ])

In [ ]:
layer_options = [
    {'conv2': 1.0},
    {'inception3a': 1.0},
    {'inception3b': 2.0},
    {'inception3b': 1.0, 'inception4a': 1.0},
    {'inception4b': 1.0, 'inception5a': 1.0},
    {'inception4c': 1.0, 'inception5b': 2.0},
    {'inception5a': 1.0},
    {'inception5b': 2.0},
    {'inception3b': 1.0, 'inception4b': 2.0, 'inception5b': 3.0},
    {'conv2': 1.0, 'inception4a': 1.0},
    {'inception4a': 1.0, 'inception5b': 3.0},
    {'inception3b': 1.0, 'inception4d': 2.0, 'inception5b': 3.0},
]

octave_options = [3, 5]
octave_scale_options = [1.2, 1.4]
learning_rates = [0.05, 0.1]
jitters = [16, 32]
optimizers = ['Adam']
iterations = 50
experiment_id = 0

In [ ]:
# exhaustive loop
for layers, num_octaves, octave_scale, lr, jitter, opt in itertools.product(
    layer_options,
    octave_options,
    octave_scale_options,
    learning_rates,
    jitters,
    optimizers
):

    experiment_id += 1
    print("\n=============================================")
    print(f"Experiment {experiment_id}")
    print(f"Layers: {layers} Num Octaves: {num_octaves}, Octave Scale: {octave_scale}, LR: {lr}, Jitter: {jitter}, Optimizer: {opt}" )
    print("=============================================\n")

    dreamed_image = deep_dream_multiscale_with_jitter(model, input_image, iterations, lr, layers, num_octaves, octave_scale, optimizer_name=opt, jitter=jitter)
    result = deprocess(dreamed_image)
    
    filename = f"dream_{experiment_id}.png"
    save_path = os.path.join("new_results", filename)
    plt.imsave(save_path, result)
    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            experiment_id,
            str(layers),
            num_octaves,
            octave_scale,
            lr,
            jitter,
            opt,
            filename
        ])
        
    plt.figure(figsize=(5, 5))
    plt.imshow(result)
    plt.title(f"Experiment {experiment_id}")
    plt.axis("off")
    plt.show()